In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# 1. Load data and reproduce the 80:20 split (Train vs. Validation/Dev set)
df = pd.read_csv('insurance_cleaned.csv')
X = df.drop(columns=['charges'])
y = df['charges']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=69
)

# 2. Define evaluated degrees and their respective optimal lambda (alpha) values
evaluated_models = [
    {'name': 'Degree 1 (OLS)', 'degree': 1, 'alpha': 0.0},
    {'name': 'Degree 2 (Lasso)', 'degree': 2, 'alpha': 17.5},
    {'name': 'Degree 3 (Lasso)', 'degree': 3, 'alpha': 12.0},
]

results = []

for item in evaluated_models:
    deg = item['degree']
    alpha = item['alpha']

    # Select estimator
    if alpha == 0.0:
        estimator = LinearRegression()
    else:
        estimator = Lasso(alpha=alpha, max_iter=5000, tol=1e-3, random_state=69)

    # Build and fit pipeline strictly on X_train
    pipe = Pipeline(
        [
            ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
            ('scaler', StandardScaler()),
            ('model', estimator),
        ]
    )

    pipe.fit(X_train, y_train)

    # Evaluate on unseen Validation set (X_val, y_val)
    y_val_pred = pipe.predict(X_val)

    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    val_r2 = r2_score(y_val, y_val_pred)

    results.append(
        {
            'Modelo': item['name'],
            'Grado Polinomio': deg,
            'Lambda (Alpha)': alpha,
            'RMSE Validación': val_rmse,
            'R² Validación': val_r2,
        }
    )

# 3. Report results in structured table
df_eval = pd.DataFrame(results)
display(df_eval)

,Modelo,Grado Polinomio,Lambda (Alpha),RMSE Validación,R² Validación
0,Degree 1 (OLS),1,0.0,6676.131631,0.715988
1,Degree 2 (Lasso),2,17.5,5537.929391,0.804574
2,Degree 3 (Lasso),3,12.0,5543.449125,0.804185
